# DIA-NN 1.9.1+ parquet spectral libraries

Since version 1.9.1, DIA-NN stores spectral libraries as parquet. AlphaBase can both **write** a library to DIA-NN's parquet format (`translate_diann.translate_to_parquet`) and **read** a DIA-NN parquet library back into a `SpecLibBase` (`reader.LibraryReaderBase`).

## Export an alphabase library to DIA-NN parquet

We build a small `SpecLibBase` (precursors + fragment intensities) and write it with `translate_to_parquet`. The output uses DIA-NN's report-style columns and `(UniMod:N)` modified sequences.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd

from alphabase.peptide.fragment import get_charged_frag_types
from alphabase.spectral_library.base import SpecLibBase
from alphabase.spectral_library.reader import LibraryReaderBase
from alphabase.spectral_library.translate_diann import translate_to_parquet

speclib = SpecLibBase(charged_frag_types=get_charged_frag_types(["b", "y"], 2))
speclib.precursor_df = pd.DataFrame(
    {
        "sequence": ["PEPTIDEK", "ACDEFGHIK"],
        "mods": ["", "Carbamidomethyl@C"],
        "mod_sites": ["", "2"],
        "charge": [2, 3],
        "rt": [0.1, 0.5],
        "proteins": ["P1", "P2"],
        "genes": ["G1", "G2"],
    }
)
speclib.calc_fragment_mz_df()
rng = np.random.default_rng(0)
speclib._fragment_intensity_df = pd.DataFrame(
    rng.random(speclib.fragment_mz_df.shape), columns=speclib.charged_frag_types
)

parquet_path = os.path.join(tempfile.mkdtemp(), "diann_lib.parquet")
translate_to_parquet(speclib, parquet_path)

pd.read_parquet(parquet_path).head()

## Read a DIA-NN parquet library

`LibraryReaderBase` recognises the `.parquet` extension and DIA-NN's column names automatically, returning a `SpecLibBase` with a `precursor_df` and dense fragment dataframes.

In [ ]:
reader = LibraryReaderBase()
reader.import_file(parquet_path)
reader.precursor_df

In [ ]:
reader.fragment_intensity_df